# Step 4 — TOD Zone Buffer Generation

Reads the finalized `tod_access_points` layer from the shared GeoPackage and
generates 200 ft, ¼ mile, and ½ mile Euclidean buffer zones around each
pedestrian access point by transit tier. Applies the SB79 policy matrix
(tier precedence, jurisdictional population thresholds, geographic scope) to
produce the final TOD Zone polygons.

> **Pipeline order:** Run after `3_tod_station_assignment_reintegration.ipynb`.
> Requires the `tod_access_points` and `tod_stations` layers to be finalized
> in the shared GeoPackage before proceeding.
> All data source paths are defined in `config.py`.


In [1]:
import pandas as pd
import geopandas as gpd
from mtcpy.census import pull_acs_data
from config import *

Info: Found credentials at: /Users/jcroff/Library/CloudStorage/Box-Box/dvutils-creds-jcroff.json


## Inputs

In [2]:
# Load Bay Area jurisdiction boundaries from the MTC ArcGIS REST service.
# Returns incorporated places and unincorporated county lands (polygon layer).
jurisdictions_gdf = gpd.read_file(JURISDICTION_BOUNDARIES_URL)
print(f"Loaded {len(jurisdictions_gdf):,} jurisdiction features")
print("CRS:", jurisdictions_gdf.crs)

# Load finalized TOD layers from the shared GeoPackage.
tod_stations_gdf = gpd.read_file(TOD_DATABASE_GPKG, layer=GPKG_TOD_STATIONS_LAYER)
print(f"Loaded {len(tod_stations_gdf):,} TOD stations")

tod_stops_gdf = gpd.read_file(TOD_DATABASE_GPKG, layer=GPKG_TOD_STOPS_LAYER)
print(f"Loaded {len(tod_stops_gdf):,} TOD stops")

tod_access_pts_gdf = gpd.read_file(TOD_DATABASE_GPKG, layer=GPKG_TOD_ACCESS_PTS_LAYER)
print(f"Loaded {len(tod_access_pts_gdf):,} TOD access points")

Loaded 109 jurisdiction features
CRS: EPSG:4326
Loaded 593 TOD stations
Loaded 755 TOD stops
Loaded 1,280 TOD access points


In [3]:
# Pull ACS 5-year (2019–2023) Total Population estimates at the Census place
# geography level via mtcpy.census.pull_acs_data.
#
# Parameters come from config.py:
#   ACS_YEAR = 2023        — endpoint year of the 5-year window
#   ACS_TYPE = "acs5"      — 5-year ACS product
#   ACS_TABLE_ID = "B01003" — Total Population table
#   ACS_GEOGRAPHY_LEVEL = "place" — incorporated places / CDPs
#
# The returned DataFrame includes:
#   B01003_001E  — total population estimate
#   NAME         — e.g. "San Jose city, California"
#   state        — FIPS state code ("06" for California)
#   place        — FIPS place code (5-digit string)
pop_df = pull_acs_data(
    year=ACS_YEAR,
    acs_type=ACS_TYPE,
    table_id=ACS_TABLE_ID,
    geography_level=ACS_GEOGRAPHY_LEVEL,
)

# Rename the estimate column for clarity and cast to integer.
pop_df = pop_df.rename(columns={"B01003_001E": "total_population"})
pop_df["total_population"] = pd.to_numeric(pop_df["total_population"], errors="coerce")

print(f"Pulled {len(pop_df):,} ACS place records")
pop_df[["jurisdiction", "total_population"]].head(5)

Pulled 101 ACS place records


,jurisdiction,total_population
0,Alameda,76876
1,Albany,19768
2,American Canyon,21698
3,Antioch,115759
4,Atherton,7021


In [4]:
# Join ACS population estimates to jurisdiction boundaries.
# Rename name columns to a shared "jurisdiction" key for consistency, then
# fix the one spelling mismatch before joining.

jurisdictions_gdf = jurisdictions_gdf.rename(columns={"jurname": "jurisdiction"})

jurisdictions_gdf["jurisdiction"] = jurisdictions_gdf["jurisdiction"].replace(
    "Saint Helena", "St. Helena"
)

jurisdictions_with_pop = jurisdictions_gdf.merge(
    pop_df[["jurisdiction", "total_population"]],
    on="jurisdiction",
    how="left",
)

jurisdictions_with_pop


,objectid,fipst,fipco,jurisdiction,Shape__Area,Shape__Length,geometry,total_population
0,219,06,097,Unincorporated Sonoma,0.450046,8.231250,"MULTIPOLYGON (((-122.62741 38.66751, -122.6238...",NaN
1,220,06,041,Unincorporated Marin,0.196976,5.260678,"MULTIPOLYGON (((-122.34747 38.07327, -122.3516...",NaN
2,221,06,055,Unincorporated Napa,0.201949,4.566464,"MULTIPOLYGON (((-122.10329 38.51335, -122.1033...",NaN
3,222,06,095,Unincorporated Solano,0.200921,6.225325,"MULTIPOLYGON (((-121.66836 38.17504, -121.6708...",NaN
4,223,06,013,Unincorporated Contra Costa,0.128281,8.168448,"MULTIPOLYGON (((-121.55701 37.81649, -121.5578...",NaN
...,...,...,...,...,...,...,...,...
104,323,06,081,San Mateo,0.004186,0.465963,"POLYGON ((-122.27668 37.53423, -122.27673 37.5...",103555.0
105,324,06,081,South San Francisco,0.007979,0.701870,"POLYGON ((-122.39193 37.67177, -122.39134 37.6...",64487.0
106,325,06,081,Woodside,0.003023,0.481513,"POLYGON ((-122.27071 37.45591, -122.2711 37.45...",5181.0
107,326,06,013,El Cerrito,0.000978,0.164531,"POLYGON ((-122.31972 37.93834, -122.31957 37.9...",25781.0


In [5]:
# Join access points → stations → stops.
#
# Relationship summary:
#   access_points → stations : many-to-one  (many access pts share one station)
#   stations      → stops    : one-to-many  (one station serves many stops)
#
# First join: left join access points to stations so every access point is
# retained; unmatched station_ids surface in the indicator check below.
stations_req_cols = ["station_id", "station_name"]
access_req_cols = ["access_id", "station_id", "access_point_name", "geometry"]
stops_req_cols = [
    "stop_id",
    "station_id",
    "stop_name",
    "route_short_name",
    "route_type",
    "tod_tier",
]

tod_access_stations_gdf = pd.merge(
    tod_access_pts_gdf[access_req_cols],
    tod_stations_gdf[stations_req_cols],
    on="station_id",
    how="left",
    indicator=True,
)

# Check: access points with a non-null station_id that didn't match a station record.
print("Access points with unmatched station_id:")
display(
    tod_access_stations_gdf[
        (tod_access_stations_gdf["_merge"] != "both")
        & (tod_access_stations_gdf["station_id"].notnull())
    ]
)

# Second join: RIGHT join to stops so all stop records are retained.
# Because stations→stops is one-to-many, a right join lets us evaluate which
# stops have no corresponding access point chain (_merge == "right_only").
tod_merged_gdf = pd.merge(
    tod_access_stations_gdf.drop(columns=["_merge"]),
    tod_stops_gdf[stops_req_cols],
    on="station_id",
    how="right",
    indicator=True,
)

# Check: stops whose station_id has no matching access point chain.
print("Stops with unmatched access point chain:")
display(
    tod_merged_gdf[(tod_merged_gdf["_merge"] != "both") & (tod_merged_gdf["station_id"].notnull())]
)

tod_merged_gdf

Access points with unmatched station_id:


,access_id,station_id,access_point_name,geometry,station_name,_merge


Stops with unmatched access point chain:


,access_id,station_id,access_point_name,geometry,station_name,stop_id,stop_name,route_short_name,route_type,tod_tier,_merge
161,NaN,mtc:santa-clara-caltrain,NaN,None,NaN,70241,Santa Clara Caltrain Station Northbound,"Limited, Local Weekday, Local Weekend",2,Tier 1,right_only
162,NaN,mtc:santa-clara-caltrain,NaN,None,NaN,70242,Santa Clara Caltrain Station Southbound,"Limited, Local Weekday, Local Weekend",2,Tier 1,right_only
221,NaN,mtc:oak,NaN,None,NaN,907401,Oakland International Airport Station,"Grey-N, Grey-S",1,Tier 1,right_only
1698,NaN,mtc:post-powell,NaN,None,NaN,16022,Post St & Powell St,2,3,2,right_only
1852,NaN,mtc:stockton-sutter,NaN,None,NaN,16524,Stockton St & Sutter St,"30, 45, 8, 8AX, 8BX, 91",3,2,right_only
1862,NaN,mtc:sutter-powell,NaN,None,NaN,16604,Sutter St & Powell St,2,3,2,right_only
2240,NaN,PS_ALUM,NaN,None,NaN,65242,Alum Rock Station,"Orange Line, OrangeE",0,Tier 2,right_only
2241,NaN,PS_ALUM,NaN,None,NaN,65243,Alum Rock Station,"Orange Line, OrangeE",0,Tier 2,right_only


,access_id,station_id,access_point_name,geometry,station_name,stop_id,stop_name,route_short_name,route_type,tod_tier,_merge
0,POWL_8,mtc:powell,B1 Market Street & 5th Street Entrance / Exit,POINT (-122.40828 37.78371),Powell,16995,Metro Powell Station/Outbound,"J, K, L, M, N",0,Tier 2,both
1,POWL_7,mtc:powell,A1 899 Market St Entrance / Exit,POINT (-122.40777 37.78458),Powell,16995,Metro Powell Station/Outbound,"J, K, L, M, N",0,Tier 2,both
2,POWL_6,mtc:powell,B2 Market Street & 5th Street (NE) Entrance / ...,POINT (-122.4074 37.7844),Powell,16995,Metro Powell Station/Outbound,"J, K, L, M, N",0,Tier 2,both
3,POWL_5,mtc:powell,B3 Market Street & 4th Street (South) Entrance...,POINT (-122.40627 37.78528),Powell,16995,Metro Powell Station/Outbound,"J, K, L, M, N",0,Tier 2,both
4,POWL_4,mtc:powell,A2 Market Street & Ellis Street Entrance / Exit,POINT (-122.40644 37.78546),Powell,16995,Metro Powell Station/Outbound,"J, K, L, M, N",0,Tier 2,both
...,...,...,...,...,...,...,...,...,...,...,...
3146,c5be4b37-713d-496d-8293-3f963f99cebc,tamien,Tamien Caltrain Station Northbound,POINT (-121.8847 37.31268),Tamien Caltrain Station,70271,Tamien Caltrain Station Northbound,"Local Weekday, Local Weekend, South County",2,Tier 1,both
3147,ecb9fe11-4f47-4cf0-a8b3-efe8e98ae2a4,tamien,Tamien Caltrain Station Northbound,POINT (-121.88571 37.31379),Tamien Caltrain Station,70271,Tamien Caltrain Station Northbound,"Local Weekday, Local Weekend, South County",2,Tier 1,both
3148,70271,tamien,Tamien Caltrain Station Northbound,POINT (-121.88479 37.31264),Tamien Caltrain Station,70272,Tamien Caltrain Station Southbound,"Local Weekday, Local Weekend, South County",2,Tier 1,both
3149,c5be4b37-713d-496d-8293-3f963f99cebc,tamien,Tamien Caltrain Station Northbound,POINT (-121.8847 37.31268),Tamien Caltrain Station,70272,Tamien Caltrain Station Southbound,"Local Weekday, Local Weekend, South County",2,Tier 1,both


In [6]:
# Filter to the working set for TOD zone generation:
#   - access points linked to a station (station_id is not null)
#   - access points with at least one corresponding stop (_merge == "both")
#
# NOTE: unmatched access points and stops are expected at this stage of review.
# The validation checks above document the gaps; this cell isolates the records
# ready for buffering.

tod_access_working_gdf = tod_merged_gdf[
    (tod_merged_gdf["_merge"] == "both")
    & (tod_merged_gdf["station_id"].notnull())
].drop(columns=["_merge"]).copy()

print(f"{len(tod_access_working_gdf):,} access point / stop combinations ready for TOD zone generation")
tod_access_working_gdf


2,594 access point / stop combinations ready for TOD zone generation


,access_id,station_id,access_point_name,geometry,station_name,stop_id,stop_name,route_short_name,route_type,tod_tier
0,POWL_8,mtc:powell,B1 Market Street & 5th Street Entrance / Exit,POINT (-122.40828 37.78371),Powell,16995,Metro Powell Station/Outbound,"J, K, L, M, N",0,Tier 2
1,POWL_7,mtc:powell,A1 899 Market St Entrance / Exit,POINT (-122.40777 37.78458),Powell,16995,Metro Powell Station/Outbound,"J, K, L, M, N",0,Tier 2
2,POWL_6,mtc:powell,B2 Market Street & 5th Street (NE) Entrance / ...,POINT (-122.4074 37.7844),Powell,16995,Metro Powell Station/Outbound,"J, K, L, M, N",0,Tier 2
3,POWL_5,mtc:powell,B3 Market Street & 4th Street (South) Entrance...,POINT (-122.40627 37.78528),Powell,16995,Metro Powell Station/Outbound,"J, K, L, M, N",0,Tier 2
4,POWL_4,mtc:powell,A2 Market Street & Ellis Street Entrance / Exit,POINT (-122.40644 37.78546),Powell,16995,Metro Powell Station/Outbound,"J, K, L, M, N",0,Tier 2
...,...,...,...,...,...,...,...,...,...,...
3146,c5be4b37-713d-496d-8293-3f963f99cebc,tamien,Tamien Caltrain Station Northbound,POINT (-121.8847 37.31268),Tamien Caltrain Station,70271,Tamien Caltrain Station Northbound,"Local Weekday, Local Weekend, South County",2,Tier 1
3147,ecb9fe11-4f47-4cf0-a8b3-efe8e98ae2a4,tamien,Tamien Caltrain Station Northbound,POINT (-121.88571 37.31379),Tamien Caltrain Station,70271,Tamien Caltrain Station Northbound,"Local Weekday, Local Weekend, South County",2,Tier 1
3148,70271,tamien,Tamien Caltrain Station Northbound,POINT (-121.88479 37.31264),Tamien Caltrain Station,70272,Tamien Caltrain Station Southbound,"Local Weekday, Local Weekend, South County",2,Tier 1
3149,c5be4b37-713d-496d-8293-3f963f99cebc,tamien,Tamien Caltrain Station Northbound,POINT (-121.8847 37.31268),Tamien Caltrain Station,70272,Tamien Caltrain Station Southbound,"Local Weekday, Local Weekend, South County",2,Tier 1


In [7]:
# Identify access points with tod_tier conflicts —
# i.e. a single access_id is associated with stops of more than one tod_tier.

tiers_per_access = tod_access_working_gdf.groupby("access_id")["tod_tier"].nunique()
tier_conflict_access_ids = tiers_per_access[tiers_per_access > 1].index

print(f"{len(tier_conflict_access_ids):,} access points have tod_tier conflicts")

conflict_mask = tod_access_working_gdf["access_id"].isin(tier_conflict_access_ids)
display_cols = ["access_id", "station_id", "station_name", "stop_id", "stop_name", "route_short_name", "route_type", "tod_tier"]

tod_tier_conflicts = (
    tod_access_working_gdf[conflict_mask][display_cols]
    .sort_values(["access_id", "tod_tier"])
)

tod_tier_conflicts

112 access points have tod_tier conflicts


,access_id,station_id,station_name,stop_id,stop_name,route_short_name,route_type,tod_tier
266,097ac7e5-554a-4441-8380-dd3477e73fd9,mtc:montgomery-bart,Montgomery BART,901202,Montgomery Street,"Blue-N, Green-N, Red-N, Yellow-N",1,Tier 1
324,097ac7e5-554a-4441-8380-dd3477e73fd9,mtc:montgomery-bart,Montgomery BART,901201,Montgomery Street,"Blue-S, Green-S, Red-S, Yellow-S",1,Tier 1
101,097ac7e5-554a-4441-8380-dd3477e73fd9,mtc:montgomery-bart,Montgomery BART,16994,Metro Montgomery Station/Outbound,"J, K, L, M, N",0,Tier 2
112,097ac7e5-554a-4441-8380-dd3477e73fd9,mtc:montgomery-bart,Montgomery BART,15731,Metro Montgomery Station/Downtown,"J, K, L, M, N",0,Tier 2
1334,097ac7e5-554a-4441-8380-dd3477e73fd9,mtc:montgomery-bart,Montgomery BART,15639,Market St & 2nd St,"7, 9, 9R, F, FBUS, KBUS, LOWL, NBUS, NOWL","0, 3",Tier 2
...,...,...,...,...,...,...,...,...
122,e3f9c1c8-f042-4ee9-84f7-40583b08ad11,mtc:mountain-view-station,Mountain View Station,64786,Mountain View Station,"Orange Line, OrangeW",0,Tier 2
372,f08dab2d-20f0-4008-af9d-f9ec67e53113,mtc:great-mall-milpitas-bart,Great Mall/Milpitas BART,909401,Milpitas,"Green-N, Orange-S",1,Tier 1
389,f08dab2d-20f0-4008-af9d-f9ec67e53113,mtc:great-mall-milpitas-bart,Great Mall/Milpitas BART,909402,Milpitas,"Green-S, Orange-N",1,Tier 1
406,f08dab2d-20f0-4008-af9d-f9ec67e53113,mtc:great-mall-milpitas-bart,Great Mall/Milpitas BART,65249,Milpitas Station,"Orange Line, OrangeE",0,Tier 2


In [8]:
# Resolve tod_tier conflicts by applying tier precedence: Tier 1 > Tier 2
# For each access point, retain only the highest-priority tier.
# Drop all stop-level fields; the result is one row per access point.

TIER_ORDER = ["Tier 1", "Tier 2"]

# Use .assign() to avoid mutating the working GDF and suppress ChainedAssignment warnings.
tod_access_pts_resolved_gdf = (
    tod_access_working_gdf
    .assign(tod_tier=pd.Categorical(tod_access_working_gdf["tod_tier"], categories=TIER_ORDER, ordered=True))
    .sort_values(["access_id", "tod_tier"])
    .drop_duplicates(subset="access_id", keep="first")
    [["access_id", "station_id", "access_point_name", "station_name", "tod_tier", "geometry"]]
    .copy()
)

# Restore tod_tier to plain string after deduplication.
tod_access_pts_resolved_gdf["tod_tier"] = tod_access_pts_resolved_gdf["tod_tier"].astype(str)

print(f"{len(tod_access_pts_resolved_gdf):,} access points ready for buffering")
print("Tier distribution:")
print(tod_access_pts_resolved_gdf["tod_tier"].value_counts().sort_index())
tod_access_pts_resolved_gdf


1,141 access points ready for buffering
Tier distribution:
tod_tier
Tier 1    393
Tier 2    726
Name: count, dtype: int64


/var/folders/9q/xt2lctm54xq6fd45m1lmgp4m0000gp/T/ipykernel_68507/420985535.py:10: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  .assign(tod_tier=pd.Categorical(tod_access_working_gdf["tod_tier"], categories=TIER_ORDER, ordered=True))


,access_id,station_id,access_point_name,station_name,tod_tier,geometry
1754,00c5cafb-2c81-48ad-983d-a45a5bef04b5,mtc:san-jose-geneva,San Jose Ave & Geneva Ave,San Jose Ave & Geneva Ave,Tier 2,POINT (-122.4465 37.72101)
1719,00c5e9fd-8f86-47a2-8344-fe1a384b9a41,mtc:row-20th,Right Of Way/20th St,Right Of Way/20th St,Tier 2,POINT (-122.42799 37.75822)
2036,01bef1b1-c456-429d-a3df-080cb5bdf485,mtc:ucsf-16th,Ucsf / Chase Center (16th St),Ucsf / Chase Center (16th St),Tier 2,POINT (-122.3893 37.76876)
2949,01c928ac-6632-4b57-a72e-180e4be4fc6a,lawrence,Lawrence Caltrain Station Northbound,Lawrence,Tier 1,POINT (-121.99607 37.37069)
510,01ce25cc-cd96-4790-86e7-e931b01b4c05,904109,Adeline Street West Entrance / Exit,Ashby,Tier 1,POINT (-122.27 37.85274)
...,...,...,...,...,...,...
175,mtc:1643701439649,mtc:fruitvale,Northbound Elevator (Platform Level),Fruitvale,Tier 1,POINT (-122.2242 37.77504)
176,mtc:1643701496687,mtc:fruitvale,Northbound Elevator (Platform Level),Fruitvale,Tier 1,POINT (-122.22435 37.77487)
172,mtc:BA-FTVL_2-1643433662776,mtc:fruitvale,Enter/Exit : Station entrance,Fruitvale,Tier 1,POINT (-122.22445 37.77508)
173,mtc:BA-FTVL_3-1643433667504,mtc:fruitvale,Enter/Exit : Station entrance,Fruitvale,Tier 1,POINT (-122.22448 37.77505)


In [9]:
# Generate non-overlapping concentric buffer rings for each access point.
#
# Band definitions:
#   inner   : 0  → 200 ft   (60.96 m)   — full inner circle
#   middle  : 200 ft → ¼ mi (402.336 m) — donut: ¼ mi circle minus 200 ft circle
#   outer   : ¼ mi → ½ mi  (804.672 m) — donut: ½ mi circle minus ¼ mi circle

BUFFER_200FT     =  60.96   # meters
BUFFER_QTR_MILE  = 402.336  # meters
BUFFER_HALF_MILE = 804.672  # meters

# 1. Project to EPSG:26910 (UTM Zone 10N) for accurate metric distances.
tod_access_pts_proj = tod_access_pts_resolved_gdf.to_crs("EPSG:26910")

# 2. Compute raw circles at each distance first, then derive donuts.
#    Circles must be saved before any subtraction — subtracting a donut instead
#    of a full circle produces incorrect geometry (e.g. half_mile would include
#    the 200 ft area again).
geom_200ft     = tod_access_pts_proj.geometry.buffer(BUFFER_200FT)
geom_qtr_mile  = tod_access_pts_proj.geometry.buffer(BUFFER_QTR_MILE)
geom_half_mile = tod_access_pts_proj.geometry.buffer(BUFFER_HALF_MILE)

rings_200ft     = tod_access_pts_proj.copy()
rings_qtr_mile  = tod_access_pts_proj.copy()
rings_half_mile = tod_access_pts_proj.copy()

rings_200ft["geometry"]     = geom_200ft
rings_qtr_mile["geometry"]  = geom_qtr_mile.difference(geom_200ft)
rings_half_mile["geometry"] = geom_half_mile.difference(geom_qtr_mile)

# 3. Tag each copy with its band label.
rings_200ft["buffer_band"]    = "200ft"
rings_qtr_mile["buffer_band"] = "quarter_mile"
rings_half_mile["buffer_band"] = "half_mile"

# 4. Stack all three bands and reproject back to EPSG:4326.
tod_buffer_rings_gdf = gpd.GeoDataFrame(
    pd.concat([rings_200ft, rings_qtr_mile, rings_half_mile], ignore_index=True),
    crs="EPSG:26910",
).to_crs("EPSG:4326")

# Verify.
expected_rows = len(tod_access_pts_resolved_gdf) * 3
assert len(tod_buffer_rings_gdf) == expected_rows, (
    f"Expected {expected_rows} rows, got {len(tod_buffer_rings_gdf)}"
)

print(f"{len(tod_buffer_rings_gdf):,} total buffer ring features ({len(tod_access_pts_resolved_gdf):,} access points × 3 bands)")
print("\nRows per band:")
print(tod_buffer_rings_gdf["buffer_band"].value_counts())
tod_buffer_rings_gdf


3,423 total buffer ring features (1,141 access points × 3 bands)

Rows per band:
buffer_band
200ft           1141
quarter_mile    1141
half_mile       1141
Name: count, dtype: int64


,access_id,station_id,access_point_name,station_name,tod_tier,geometry,buffer_band
0,00c5cafb-2c81-48ad-983d-a45a5bef04b5,mtc:san-jose-geneva,San Jose Ave & Geneva Ave,San Jose Ave & Geneva Ave,Tier 2,"POLYGON ((-122.44581 37.72101, -122.44581 37.7...",200ft
1,00c5e9fd-8f86-47a2-8344-fe1a384b9a41,mtc:row-20th,Right Of Way/20th St,Right Of Way/20th St,Tier 2,"POLYGON ((-122.42729 37.75822, -122.4273 37.75...",200ft
2,01bef1b1-c456-429d-a3df-080cb5bdf485,mtc:ucsf-16th,Ucsf / Chase Center (16th St),Ucsf / Chase Center (16th St),Tier 2,"POLYGON ((-122.38861 37.76876, -122.38861 37.7...",200ft
3,01c928ac-6632-4b57-a72e-180e4be4fc6a,lawrence,Lawrence Caltrain Station Northbound,Lawrence,Tier 1,"POLYGON ((-121.99538 37.37069, -121.99539 37.3...",200ft
4,01ce25cc-cd96-4790-86e7-e931b01b4c05,904109,Adeline Street West Entrance / Exit,Ashby,Tier 1,"POLYGON ((-122.26931 37.85273, -122.26931 37.8...",200ft
...,...,...,...,...,...,...,...
3418,mtc:1643701439649,mtc:fruitvale,Northbound Elevator (Platform Level),Fruitvale,Tier 1,"POLYGON ((-122.21511 37.77427, -122.21525 37.7...",half_mile
3419,mtc:1643701496687,mtc:fruitvale,Northbound Elevator (Platform Level),Fruitvale,Tier 1,"POLYGON ((-122.21526 37.7741, -122.2154 37.773...",half_mile
3420,mtc:BA-FTVL_2-1643433662776,mtc:fruitvale,Enter/Exit : Station entrance,Fruitvale,Tier 1,"POLYGON ((-122.21536 37.77431, -122.2155 37.77...",half_mile
3421,mtc:BA-FTVL_3-1643433667504,mtc:fruitvale,Enter/Exit : Station entrance,Fruitvale,Tier 1,"POLYGON ((-122.21539 37.77428, -122.21553 37.7...",half_mile


In [10]:
import tempfile, webbrowser

BAND_COLORS = {
    "200ft": "#039438",  # green
    "quarter_mile": "#ff7f00",  # orange
    "half_mile": "#377eb8",  # blue
}

TEST_STATION_ID = "mtc:san-jose-geneva"

# Layer 1: buffer rings, colored by band.
m = tod_buffer_rings_gdf[tod_buffer_rings_gdf["station_id"] == TEST_STATION_ID].explore(
    column="buffer_band",
    categorical=True,
    cmap=[BAND_COLORS[b] for b in ["200ft", "quarter_mile", "half_mile"]],
)

# Layer 2: access points for the same station, added to the existing map.
tod_access_pts_resolved_gdf[tod_access_pts_resolved_gdf["station_id"] == TEST_STATION_ID].explore(
    m=m,
    color="black",
    marker_kwds={"radius": 6},
    tooltip=["access_id", "access_point_name", "tod_tier"],
)

tmp = tempfile.NamedTemporaryFile(suffix=".html", delete=False)
m.save(tmp.name)
webbrowser.open(f"file://{tmp.name}")
print(f"Map saved to: {tmp.name}")

Map saved to: /var/folders/9q/xt2lctm54xq6fd45m1lmgp4m0000gp/T/tmpi842yc1u.html


In [12]:
# ── Dissolve tod_buffer_rings_gdf by tod_tier + buffer_band ──────────────────
import uuid

# 1. Reproject to EPSG:26910 for accurate geometry operations.
tod_buffer_rings_proj = tod_buffer_rings_gdf.to_crs("EPSG:26910")

# 2. Dissolve by tod_tier + buffer_band — merges overlapping per-access-point
#    circles from adjacent access points into one continuous shape per tier + band.
tod_zones_dissolved = tod_buffer_rings_proj.dissolve(
    by=["tod_tier", "buffer_band"],
    as_index=False,
)[["tod_tier", "buffer_band", "geometry"]]

# 3. Difference the dissolved circles per tod_tier to produce clean non-overlapping rings.
rows = []
for tier, group in tod_zones_dissolved.groupby("tod_tier"):
    geom_lookup = group.set_index("buffer_band")["geometry"]

    ring_200ft     = geom_lookup["200ft"]
    ring_qtr_mile  = geom_lookup["quarter_mile"].difference(geom_lookup["200ft"])
    ring_half_mile = geom_lookup["half_mile"].difference(geom_lookup["quarter_mile"])

    for band, geom in [("200ft", ring_200ft), ("quarter_mile", ring_qtr_mile), ("half_mile", ring_half_mile)]:
        rows.append({"zone_id": str(uuid.uuid4()), "tod_tier": tier, "buffer_band": band, "geometry": geom})

tod_zones_gdf = gpd.GeoDataFrame(rows, crs="EPSG:26910").to_crs("EPSG:4326")

print(f"{len(tod_zones_gdf)} TOD zones generated:")
print(tod_zones_gdf[["zone_id", "tod_tier", "buffer_band"]])

# 4. Map — full Bay Area dissolved rings + access points.
#    Zoom in on the browser to inspect individual stations.
m = tod_zones_gdf.explore(
    column="buffer_band",
    categorical=True,
    cmap=[BAND_COLORS[b] for b in ["200ft", "quarter_mile", "half_mile"]],
    tooltip=["zone_id", "tod_tier", "buffer_band"],
    style_kwds={"fillOpacity": 0.4, "weight": 1.5},
)

tod_access_pts_resolved_gdf.explore(
    m=m,
    color="black",
    marker_kwds={"radius": 3},
    tooltip=["access_id", "access_point_name", "tod_tier"],
)

tmp = tempfile.NamedTemporaryFile(suffix=".html", delete=False)
m.save(tmp.name)
webbrowser.open(f"file://{tmp.name}")
print(f"Map saved to: {tmp.name}")

6 TOD zones generated:
                                zone_id tod_tier   buffer_band
0  80afdc1e-75d1-402e-abf6-3865998e6c23   Tier 1         200ft
1  810c2af6-2fae-43d3-ad44-4ea4339103e7   Tier 1  quarter_mile
2  5b695b03-a519-4308-8542-027ba5f1ddc5   Tier 1     half_mile
3  9ec14395-7dcd-45e0-9375-222fdd7dc031   Tier 2         200ft
4  d9c5a9cb-9a8f-4d7a-a310-63ee269d315e   Tier 2  quarter_mile
5  9a758e96-6f8d-459d-9076-67f666849e10   Tier 2     half_mile
Map saved to: /var/folders/9q/xt2lctm54xq6fd45m1lmgp4m0000gp/T/tmpalogulzr.html
